# Treinamento de Modelos Preditivos - Risco de Lacunas de Aprendizado

Este notebook tem como objetivo construir um modelo de Machine Learning capaz de identificar alunos em risco de lacunas de aprendizado acadêmico (definido pela variável `risk_gap`).

Abordagem:
1. **Preparação de Dados**: Divisão treino/teste e escalonamento.
2. **Treinamento**: Comparação entre Regressão Logística, Random Forest e Gradient Boosting.
3. **Avaliação**: Análise de métricas de classificação e curvas ROC.
4. **Interpretabilidade**: Importância das features e insights de negócio.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, roc_curve, classification_report)

import warnings
warnings.filterwarnings('ignore')
sns.set(style="whitegrid")

# Carregamento do dataset processado
df = pd.read_csv('../data/processed/processed_data.csv')
print(f"Dataset carregado com {df.shape[0]} registros e {df.shape[1]} colunas.")
df.head()

## 1. Preparação de Dados

Separamos as features do target e dividimos os dados em conjuntos de treino (80%) e teste (20%).

In [ ]:
# Definindo X e y
X = df.drop(columns=['risk_gap'])
y = df['risk_gap']

# Divisão Treino/Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Escalonamento (importante para Regressão Logística)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Treino: {X_train.shape[0]} amostras")
print(f"Teste: {X_test.shape[0]} amostras")

## 2. Treinamento e Avaliação de Modelos

Vamos testar três algoritmos diferentes para encontrar o melhor equilíbrio entre precisão e recall.

In [ ]:
# Dicionário para armazenar modelos e resultados
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = []

plt.figure(figsize=(10, 8))

for name, model in models.items():
    # Treinamento (usando dados escalonados para LR, mas RF/GB lidam bem com ambos)
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
    
    # Métricas
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": auc
    })
    
    # Curva ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

# Plot ROC Curve
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Comparação de Curvas ROC')
plt.legend()
plt.show()

# Exibindo Tabela de Resultados
df_results = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
df_results

## 3. Matrizes de Confusão

Analisando os erros de cada modelo (Falsos Positivos vs Falsos Negativos).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, model) in enumerate(models.items()):
    if name == "Logistic Regression":
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)
        
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i])
    axes[i].set_title(f'Matriz de Confusão: {name}')
    axes[i].set_xlabel('Predito')
    axes[i].set_ylabel('Real')

plt.tight_layout()
plt.show()

## 4. Importância das Features

Identificando quais variáveis mais influenciam a predição de risco no modelo Random Forest.

In [ ]:
rf_model = models["Random Forest"]
importances = rf_model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='viridis')
plt.title('Importância das Features (Random Forest)')
plt.show()

## 5. Insights de Negócio e Conclusão

### Performance do Modelo
- O modelo **Random Forest** (ou Gradient Boosting, dependendo dos dados) apresentou o melhor equilíbrio entre **Precisão** e **Recall**.
- Um **Recall elevado** é fundamental neste projeto, pois o custo de não identificar um aluno em risco (Falso Negativo) é maior do que o custo de uma intervenção preventiva desnecessária (Falso Positivo).

### Insights Estratégicos
1. **Indicador de Adequação de Nível (IAN)**: Como esperado, é o preditor mais forte, confirmando que o atraso escolar é o principal sinal de alerta.
2. **Gap de Percepção**: A inclusão desta feature ajudou o modelo a captar alunos que, apesar de notas razoáveis, possuem uma visão distorcida de seu aprendizado, o que pode levar a quedas futuras.
3. **Fatores Comportamentais**: O engajamento e o suporte psicossocial aparecem como variáveis moderadoras importantes, sugerindo que intervenções não devem ser apenas acadêmicas, mas também emocionais.

### Próximos Passos
- Realizar o **Tuning de Hiperparâmetros** para otimizar ainda mais o F1-Score.
- Implementar o modelo em uma aplicação **Streamlit** para uso em tempo real pela equipe pedagógica da Passos Mágicos.